In [0]:
from uuid import uuid4

from pyspark.sql import functions as F
from pyspark.sql import types as T
respath = "development_042_silver_sandbox.demand_forecast"

THEME_MAPPING_TABLE = (
    f"{respath}.game_taxonomy_unification_v1_theme_mapping"
)
MECHANIC_MAPPING_TABLE = (
    f"{respath}.game_taxonomy_unification_v1_mechanic_mapping"
)

# Database containing the extraction and final output tables
RES_PATH = "development_042_silver_sandbox.demand_forecast"

# Notebook 2 output: read-only input for this notebook
SOURCE_TABLE = (
    f"{RES_PATH}.overview_ctc_line_reel_theme_extracted_v1"
)

# Run metadata
RUN_ID = str(uuid4())
TAXONOMY_VERSION = "v1"

# Raw per-game assignments: preserve each extracted label, score/status,
# definition, and evidence associated with each game.
RAW_THEME_ASSIGNMENTS_TABLE = (
    f"{RES_PATH}.overview_ctc_line_reel_theme_raw_assignments_{TAXONOMY_VERSION}"
)
RAW_MECHANIC_ASSIGNMENTS_TABLE = (
    f"{RES_PATH}.overview_ctc_line_reel_mechanic_raw_assignments_{TAXONOMY_VERSION}"
)

# Compact raw definition tables for reviewing and unifying labels.
RAW_THEME_DEF_TABLE = (
    f"{RES_PATH}.overview_ctc_line_reel_theme_raw_def_{TAXONOMY_VERSION}"
)
RAW_MECHANIC_DEF_TABLE = (
    f"{RES_PATH}.overview_ctc_line_reel_mechanic_raw_def_{TAXONOMY_VERSION}"
)

# Reviewed raw-to-canonical mappings and canonical dictionaries.
THEME_MAPPING_TABLE = (
    f"{RES_PATH}.overview_ctc_line_reel_theme_mapping_{TAXONOMY_VERSION}"
)
MECHANIC_MAPPING_TABLE = (
    f"{RES_PATH}.overview_ctc_line_reel_mechanic_mapping_{TAXONOMY_VERSION}"
)
THEME_DICTIONARY_TABLE = (
    f"{RES_PATH}.overview_ctc_line_reel_theme_dictionary_{TAXONOMY_VERSION}"
)
MECHANIC_DICTIONARY_TABLE = (
    f"{RES_PATH}.overview_ctc_line_reel_mechanic_dictionary_{TAXONOMY_VERSION}"
)

# Canonical assignments, including per-game evidence and scores/status.
CANONICAL_THEME_ASSIGNMENTS_TABLE = (
    f"{RES_PATH}.overview_ctc_line_reel_theme_assignments_{TAXONOMY_VERSION}"
)
CANONICAL_MECHANIC_ASSIGNMENTS_TABLE = (
    f"{RES_PATH}.overview_ctc_line_reel_mechanic_assignments_{TAXONOMY_VERSION}"
)

# Build and inspect this candidate before publishing the final table.
CANDIDATE_GAME_TABLE = (
    f"{RES_PATH}.game_attr_ai_bronz_candidate_{TAXONOMY_VERSION}"
)
FINAL_GAME_TABLE = f"{RES_PATH}.game_attr_ai_bronz"

print(f"Source: {SOURCE_TABLE}")
print(f"Run ID: {RUN_ID}")
print(f"Taxonomy version: {TAXONOMY_VERSION}")
print(f"Candidate game table: {CANDIDATE_GAME_TABLE}")
print(f"Final game table: {FINAL_GAME_TABLE}")

In [0]:
source_df = spark.table(SOURCE_TABLE)

required_columns = {"game_name", "themes", "mechanics"}
missing_columns = required_columns - set(source_df.columns)

if missing_columns:
    raise ValueError(
        f"Source table is missing required columns: {sorted(missing_columns)}"
    )

blank_game_names = source_df.where(
    F.col("game_name").isNull()
    | (F.trim(F.col("game_name")) == "")
).count()

if blank_game_names:
    raise ValueError(
        f"Source table contains {blank_game_names} rows with a blank game_name."
    )

row_count = source_df.count()

duplicate_game_names = (
    source_df
    .groupBy(F.lower(F.trim(F.col("game_name"))).alias("_game_key"))
    .count()
    .where(F.col("count") > 1)
    .count()
)

print(f"Source table: {SOURCE_TABLE}")
print(f"Rows: {row_count}")
print(f"Repeated normalized game names: {duplicate_game_names}")

source_df.printSchema()
display(source_df.limit(5))

In [0]:
from pyspark.sql import functions as F


source_for_normalization_df = spark.table(SOURCE_TABLE)

if "game_name" not in source_for_normalization_df.columns:
    raise ValueError(
        f"Source table {SOURCE_TABLE} does not contain a game_name column."
    )

blank_game_names = source_for_normalization_df.where(
    F.col("game_name").isNull()
    | (F.trim(F.col("game_name")) == "")
).count()

if blank_game_names:
    raise ValueError(
        f"Source table contains {blank_game_names} rows with a blank game_name."
    )

normalized_source_df = (
    source_for_normalization_df
    .withColumn("game_name_raw", F.col("game_name"))
    .withColumn(
        "game_name_key",
        F.regexp_replace(
            F.lower(F.trim(F.col("game_name"))),
            r"\s+",
            " "
        )
    )
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("taxonomy_version", F.lit(TAXONOMY_VERSION))
)

duplicate_game_keys_df = (
    normalized_source_df
    .groupBy("game_name_key")
    .count()
    .where(F.col("count") > 1)
    .orderBy(F.desc("count"), "game_name_key")
)

print(f"Rows normalized: {normalized_source_df.count()}")
print(f"Repeated normalized game keys: {duplicate_game_keys_df.count()}")

display(duplicate_game_keys_df)
display(normalized_source_df.limit(5))

In [0]:
from pyspark.sql import DataFrame


theme_evidence_schema = T.ArrayType(
    T.StructType([
        T.StructField("page_number", T.IntegerType(), True),
        T.StructField("source_type", T.StringType(), True),
        T.StructField("excerpt_or_visual_description", T.StringType(), True),
    ])
)

theme_array_schema = T.ArrayType(
    T.StructType([
        T.StructField("name", T.StringType(), True),
        T.StructField("definition", T.StringType(), True),
        T.StructField("score", T.DoubleType(), True),
        T.StructField("evidence", theme_evidence_schema, True),
    ])
)


def as_theme_array(df: DataFrame, column_name: str):
    """Return a theme column as an array, parsing it if it is JSON text."""
    data_type = df.schema[column_name].dataType

    if isinstance(data_type, T.ArrayType):
        return F.col(column_name)
    if isinstance(data_type, T.StringType):
        return F.from_json(F.col(column_name), theme_array_schema)

    raise TypeError(
        f"Expected {column_name} to be an array or JSON string; "
        f"found {data_type.simpleString()}."
    )


raw_theme_assignments_df = (
    normalized_source_df
    .withColumn("_themes_array", as_theme_array(normalized_source_df, "themes"))
    .withColumn("_theme", F.explode("_themes_array"))
    .select(
        "game_name",
        "game_name_raw",
        "game_name_key",
        F.col("_theme.name").alias("raw_theme_name"),
        F.regexp_replace(
            F.lower(F.trim(F.col("_theme.name"))), r"\s+", " "
        ).alias("raw_theme_name_key"),
        F.col("_theme.definition").alias("raw_theme_definition"),
        F.col("_theme.score").cast("double").alias("raw_theme_score"),
        F.col("_theme.evidence").alias("evidence"),
        "run_id",
        "taxonomy_version",
    )
)

raw_theme_def_df = (
    raw_theme_assignments_df
    .groupBy(
        "raw_theme_name",
        "raw_theme_name_key",
        "raw_theme_definition",
    )
    .agg(
        F.countDistinct("game_name_key").alias("game_count"),
        F.first("evidence", ignorenulls=True).alias("sample_evidence"),
    )
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("taxonomy_version", F.lit(TAXONOMY_VERSION))
)

(
    raw_theme_assignments_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(RAW_THEME_ASSIGNMENTS_TABLE)
)

(
    raw_theme_def_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(RAW_THEME_DEF_TABLE)
)

print(f"Raw theme assignments: {RAW_THEME_ASSIGNMENTS_TABLE}")
print(f"Raw theme definitions: {RAW_THEME_DEF_TABLE}")
print(f"Assignment rows: {raw_theme_assignments_df.count()}")
print(f"Distinct raw name-definition pairs: {raw_theme_def_df.count()}")

display(raw_theme_def_df.orderBy("raw_theme_name", "raw_theme_definition"))

In [0]:
mechanic_evidence_schema = T.ArrayType(
    T.StructType([
        T.StructField("page_number", T.IntegerType(), True),
        T.StructField("source_type", T.StringType(), True),
        T.StructField("excerpt_or_visual_description", T.StringType(), True),
    ])
)

mechanic_array_schema = T.ArrayType(
    T.StructType([
        T.StructField("name", T.StringType(), True),
        T.StructField("definition", T.StringType(), True),
        T.StructField("score", T.DoubleType(), True),
        T.StructField("evidence", mechanic_evidence_schema, True),
    ])
)


def as_mechanic_array(df, column_name):
    """Return a mechanic column as an array, parsing it if it is JSON text."""
    data_type = df.schema[column_name].dataType

    if isinstance(data_type, T.ArrayType):
        return F.col(column_name)
    if isinstance(data_type, T.StringType):
        return F.from_json(F.col(column_name), mechanic_array_schema)

    raise TypeError(
        f"Expected {column_name} to be an array or JSON string; "
        f"found {data_type.simpleString()}."
    )


raw_mechanic_assignments_df = (
    normalized_source_df
    .withColumn(
        "_mechanics_array",
        as_mechanic_array(normalized_source_df, "mechanics")
    )
    .withColumn("_mechanic", F.explode("_mechanics_array"))
    .select(
        "game_name",
        "game_name_raw",
        "game_name_key",
        F.col("_mechanic.name").alias("raw_mechanic_name"),
        F.regexp_replace(
            F.lower(F.trim(F.col("_mechanic.name"))), r"\s+", " "
        ).alias("raw_mechanic_name_key"),
        F.col("_mechanic.definition").alias("raw_mechanic_definition"),
        F.col("_mechanic.score").cast("double").alias("raw_mechanic_score"),
        F.col("_mechanic.evidence").alias("evidence"),
        "run_id",
        "taxonomy_version",
    )
)

raw_mechanic_def_df = (
    raw_mechanic_assignments_df
    .groupBy(
        "raw_mechanic_name",
        "raw_mechanic_name_key",
        "raw_mechanic_definition",
    )
    .agg(
        F.countDistinct("game_name_key").alias("game_count"),
        F.first("evidence", ignorenulls=True).alias("sample_evidence"),
    )
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("taxonomy_version", F.lit(TAXONOMY_VERSION))
)

(
    raw_mechanic_assignments_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(RAW_MECHANIC_ASSIGNMENTS_TABLE)
)

(
    raw_mechanic_def_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(RAW_MECHANIC_DEF_TABLE)
)

print(f"Raw mechanic assignments: {RAW_MECHANIC_ASSIGNMENTS_TABLE}")
print(f"Raw mechanic definitions: {RAW_MECHANIC_DEF_TABLE}")
print(f"Mechanic assignment rows: {raw_mechanic_assignments_df.count()}")

print("Raw themes — review names, definitions, counts, and sample evidence")
display(
    spark.table(RAW_THEME_DEF_TABLE)
    .orderBy("raw_theme_name", "raw_theme_definition")
)

print("Raw mechanics — review names, definitions, counts, and sample evidence")
display(
    spark.table(RAW_MECHANIC_DEF_TABLE)
    .orderBy("raw_mechanic_name", "raw_mechanic_definition")
)

In [0]:
"""Draft raw-label-to-canonical mappings generated from the attached inventories.
Review before writing final mapping tables.
"""
THEME_CANONICAL_OVERRIDES = {
    "Abstract and Colors": ["colors"],
    "African Safari / Wildlife": [
        "African Culture and Wildlife", "African Safari", "Jungle",
        "Jungle/Amazon", "Jungle/Wildlife", "Safari",
    ],
    "Ancient Civilizations": [
        "Ancient", "ancient", "Ancient City", "Ancient Culture",
        "ancient or historical figures", "Ancient or Mythological",
        "Ancient Temple", "Ancient Warfare", "Angkor Wat", "Aztec",
        "empire", "knight", "Mayan", "Medieval", "Medieval Fantasy",
        "medieval fantasy", "Medieval/Fantasy", "Native American",
        "Prehistoric", "prehistoric", "Shrine", "Temple",
        "Wild West/Native American",
    ],
    "Ancient Egypt": [
        "Ancient Egypt", "ancient egypt", "Ancient Egyptian",
        "Ancient or Egyptian", "Ancient or Egyptian Theme",
        "ancient_egypt", "Egyptian", "egyptian",
    ],
    "Ancient Greece and Rome": [
        "Ancient Greece", "Ancient Greece or Rome",
        "Ancient Greece/Rome", "Greek mythology",
    ],
    "Animals and Wildlife": [
        "Animal", "Animal Theme", "Animals", "animals",
        "Aquatic", "aquatic", "Australian Animals", "Australian animals",
        "barnyard", "Bees and Honey", "Birds", "Birds or Nature",
        "Butterfly", "cartoon animal", "Cat", "Cats", "Dairy Farm",
        "Dinosaurs", "Dogs", "Dragons", "Ducks", "equine", "Falcon",
        "Farm", "Farm or Animal", "farm_animals", "Fishing", "horse",
        "Ladybug", "Lobster", "Marine", "Marine Life", "Ocean", "ocean",
        "Ocean/Lobster", "otter", "Panther", "Peacock", "Pets", "Piggy",
        "Poodles", "Raven", "Snakes", "Spider", "Turtle", "Underwater",
        "underwater", "Underwater/Beach", "Underwater/Ocean", "Wild Cats",
        "wild_animals", "Wildlife", "wildlife", "Wildlife or Animal Theme",
        "Wildlife Theme", "Wildlife/Animals", "Wildlife/Buffalo",
        "Wildlife/Nature", "wolf",
    ],
    "Art and Literature": [
        "Art", "Art/Artistic", "Art/Leonardo", "Art/painting",
        "card_suits", "Cards", "cards", "Gypsy", "Hearts",
        "Historical Figures", "Literary", "literary", "opera",
        "Sports Party", "Warriors",
    ],
    "Asian Culture": [
        "Ancient China", "Asian", "asian", "Asian animals",
        "Asian Cultural", "Asian Cultural Elements",
        "Asian Cultural Theme", "Asian Culture", "Asian Dragon",
        "Asian Fantasy", "Asian Floral", "Asian Garden", "Asian motifs",
        "Asian Mythology", "Asian Theme", "Asian Treasure",
        "Asian-themed", "Asian/Japanese Culture", "asian_culture",
        "Buddha", "buddha or eastern spirituality",
        "Chinese Cultural Elements", "Chinese Culture",
        "Chinese mythology", "Chinese New Year", "chinese_culture",
        "chinese_mythology", "dragon and gemstones", "Dragons",
        "Japanese Culture", "japanese culture", "Koi", "Lunar",
        "Mythical", "mythological creatures", "Oriental", "oriental",
        "Panda", "Phoenix", "Royal/Majesty", "Samurai", "Tigers",
        "Tigers and Jewels", "Totem", "Wild Cats", "Wild West/Wolves",
    ],
    "Buffalo": ["Wildlife/Buffalo"],
    "Cards and Table Games": ["Dice", "dice", "Poker", "poker", "Poker Theme", "Roulette"],
    "Cats": ["Cat", "Cats", "Lucky Cat", "Wild Cats"],
    "Classic Slots and Symbols": [
        "7s", "Arcade", "Bar", "Bars", "bars", "Bars and 7s",
        "bars and bells", "Bells", "Casino", "Casino Gambling",
        "Casino Night", "Classic", "Classic Slots", "Classic Theme",
        "gold bar 7s", "Gold Bars", "Pinball", "pinball", "Retro",
        "Sevens", "sevens", "Sevens and Bars", "Vegas", "Wheels", "Wild",
        "Wild West",
    ],
    "Culture and People": [
        "Arctic", "arctic", "australia", "australian", "Irish",
        "Leprechaun or Irish Theme", "mexican", "Patriotic", "patriotic",
        "russian dolls",
    ],
    "Dogs": ["Dogs", "Poodles"],
    "Fantasy and Magic": [
        "Fairy", "Fairy Tale", "Fantasy", "fantasy", "fantasy_creatures",
        "Hexbreaker", "Magic", "magic", "Middle Eastern/Fantasy",
        "Monsters", "Space Fantasy", "supernatural", "vampire", "Winter",
        "Witch or Fantasy", "Wizard",
    ],
    "Flowers and Garden": ["Flowers", "Garden", "garden", "Rose"],
    "Food and Drink": [
        "Bar or Tavern", "Beans", "Candy", "Chili", "Classic Fruit",
        "Food", "food", "food or spice theme", "Fruit", "fruit",
        "Fruit Theme", "Fruit theme", "Fruit/Bar", "Fruit/vegetable",
        "Gemstones", "Jalapeno", "red hot tamales", "retro_fruit",
    ],
    "Holidays and Festivals": [
        "Birthday", "Cultural/Festival", "Festival", "festival", "Fiesta",
        "Fire", "Firecracker", "fireworks", "Fireworks/Rockets",
        "Halloween", "halloween", "Masquerade",
    ],
    "Home and Lifestyle": ["Home Decor"],
    "Luck and Zodiac": [
        "Bad Luck", "Clover", "clover", "Irish Luck", "Luck",
        "Lucky 7s", "Lucky Cat", "Lucky Charms", "Lucky Clover",
        "Lucky Ox", "lucky_seven", "Zodiac",
    ],
    "Marine and Ocean": [
        "Aquatic", "aquatic", "beach", "Beach or Surfing", "Fishing",
        "island", "Lobster", "Marine", "Marine Life", "nautical",
        "Ocean", "ocean", "Ocean/Lobster", "Summer/Beach", "Tropical",
        "tropical", "Tropical Island", "tropical island", "Tropical Theme",
        "Tropical/Lagoon", "Underwater", "underwater", "Underwater/Beach",
        "Underwater/Ocean",
    ],
    "Music and Dance": [
        "Adventure", "Ballet", "dance", "Disco", "Latin", "Latin/Salsa",
        "Music", "music", "Music or Dance", "Salsa",
    ],
    "Mystery and Detective": ["Crime", "Detective", "Heist", "Lock and Key", "Mystery"],
    "Mythology": [
        "Celtic", "Dragons", "Genie", "Mythical Creature",
        "mythical creature", "Mythical Creatures", "mythical_creatures",
        "mythological", "Mythological or Supernatural", "Mythology",
        "mythology", "Norse", "Olympus", "viking",
    ],
    "Nature and Seasons": [
        "adult", "alpine", "explosion", "Explosive",
        "farm or agriculture", "Jewel-themed", "Nature", "nature",
        "Nature/Wildlife", "Rural", "Storm", "Vivaldi's Seasons", "Weather",
    ],
    "Other Theme": [
        "Acorn", "Bubbles", "Bullseye Bonanza", "Camping", "characters",
        "Leprechaun", "Southern", "Warrior", "warrior", "wheel",
    ],
    "Pirates and Nautical": ["nautical", "Pirate", "Pirates"],
    "Pop Culture": ["Black Widow", "Ghostbusters", "I Dream of Jeannie"],
    "Royalty and Romance": [
        "Beauty", "demolition", "Female", "female_adult", "Maid",
        "Mechanical", "Princess", "Royal or Imperial", "royalty",
        "Southern Belle", "Wedding",
    ],
    "Science Fiction": ["Cosmic", "Science Fiction", "Space", "Space Fantasy"],
    "Sports": ["American Football", "Golf", "Racing", "Shooting Gallery", "Strike"],
    "Technology and Industry": ["Hardware Store", "Steam"],
    "Toys and Cartoon Worlds": ["Toy"],
    "Travel and Exploration": [
        "Arctic/Adventure", "Aviation", "Country Club", "Exploration",
        "Transportation", "travel", "Urban", "urban",
    ],
    "Treasure and Wealth": [
        "Cash", "cash", "Charms", "Coins", "Currency", "Diamond",
        "diamond", "Diamonds", "diamonds", "Dollars", "Fortune", "Gems",
        "Gold", "Gold Bars", "Gold Doubloon", "Jewel",
        "Jewel or Gem Theme", "Jewel or Treasure", "Jewel/precious stone",
        "Jewelry", "jewelry", "Jewelry/Mansion", "Jewels", "jewels",
        "Jewels/Gems", "Luxury", "luxury", "Mining", "Mining or Treasure",
        "Money", "Pearl", "Piggy Bank", "Pot of Gold", "Riches", "royal",
        "royal_jewels", "Shopping", "Treasure", "treasure",
        "Treasure/Pearl", "Treasure Hunt", "treasure hunt",
        "Treasure or Wealth", "Treasure Theme", "Triple Diamond", "Wealth",
        "wealth", "Wealth or Prosperity",
    ],
    "Vehicles and Aviation": ["Carnival", "fair"],
    "Western and Frontier": ["Western", "western", "Western/Oil Drilling"],
}

THEME_DROP_KEYS = ["Multiple Game Themes"]

### MECHS
MECHANIC_CANONICAL_OVERRIDES = {
    "Card / Table Game Actions": """6 Card Charlie|AutoHold|Autohold|Betting Types|Card Draw|Card Drawing Rules|Card Holding Mechanism|Card Replacement|Commission on Banker Hand|Double Down|Dream Card Feature|Extra Cards|Hold Feature|Insurance|Multi-Game Configuration|Multi-Game Poker|Multi-Hand Blackjack|Multi-Hand Draw Poker|Multi-Hand Feature|Multi-Hand Play|Multi-Hand Poker|Multi-Play Poker|Multi-game selection|Multi-hand|Multi-hand Play|Multi-hand Poker|Multi-hand Video Poker|Multi-hand play|Other Games|Paytable Configuration|Player-selectable card flip option|Poker|Poker Game Variations|Poker Gameplay|Poker Hand Evaluation|Poker Variants|Poker Variations|Quick Quads Feature|Roulette Chip Management|Single-Hand Autostart|Split|Split Card|Super Hand Feature|Surrender|Top Line Bet|Triple Play|autohold|multi-hand|multi_hand|nudge|split card""".split("|"),

    "Coin / Credit Collection": """Cash on Reels|Nautilus Coin Feature|coin_feature|Ca$h Wilds Meter""".split("|"),

    "Expanding / Transforming Reels": """Coin Boost and Reel Boost Super Feature|Cooperative Symbols Feature|Dancing Reels Feature|Disc Feature|End2End feature|Expanding Fortune Spin|Expanding Reel Feature|Expanding Reels Feature|Expanding Symbol Feature|Expanding Symbols|Reel Boost|Reel Expansion|Reel Transformation|Replicating Reels Feature|Rift Symbols Feature|Shifting Sands Feature|Star Energizer Feature|Star Expansion Feature|Super Stacks Bonus|Transformation Feature|Ultra Plus Bonus|Unicorn Feature|expanding reels feature|Double Reels""".split("|"),

    "Free Spins / Free Games": """ADVENTURE Feature|Additional Free Spins|Bobber Bonus Free Games|Bonus Choice|Bonus Collide Feature|Bonus Craze|Bonus Free Games|Bonus Pick|Bonus Select|Bonus Spins|Bonus Wilds|Burning Wild Multipliers|Ca$h Wilds Meter|Cai Shen Free Spins Feature|Cai Shen Wheel Feature|Cash on Reels Feature|Cherished Free Games|Choose Your Destiny Free Games Bonus|Clue Collection Bonus|Coin Boost Feature|Congo Free Games|Congo Multiplier Wheel|Countdown Wilds Feature|Deluxe Free Games|Double Reels|Expanded Reels Feature|Expanding Free Games|Expanding Wild|Extra Paylines Bonus|Extra Wilds|FORTUNE Feature|Fate Changer Bonuses|Fiesta Free Games Bonus|Fire Horse Bonus|Fortune Charm Feature|Fortune Trader Bonus|Fortune Wild Feature|Freddy's Free Games Bonus|Free Game Bonus|Free Games|Free Games Bonus|Free Games Bonus Feature|Free Games Falls Bonus|Free Games Feature|Free Games Multiplier|Free Games Retrigger|Free Games Trigger|Free Games Wheel Feature|Free Games Wins Multiplier|Free Games bonus|Free Games feature|Free Spins|Free Spins Bonus|Fruitastic Bonus|Gem Feature|Glorious Tiger Free Games|Gold Peach Respins Feature|Gold Spins Feature|Gold Stallion Respins Feature|Golden Pepper Free Games|Golden Staff Wilds and Multipliers|Golden Sun Feature|Hummingbird Bonus Feature|Jeannie's Blink of an Eye Bonus|Jeannie's Treasure Chest Bonus|Jumping Wilds Feature|Keys to Riches Free Games|Kitty Glitter Bonus|Legendary Free Games Bonus|Lightning Wilds Feature|Locking Multiplier|Locking Wilds|Masters Bonus|Move-Expand Stacks|Multiplier Feature|Multiplier Free Spins|Multiplier Hotspots|Multiplier Madness Bonus|Multiplier on Reel 1|Multiply Bonus|Multiply Deluxe Bonus|Mystery Bonus|Mystery Stacks|Mystery Symbols|Mystical Mermaid Bonus|Nudge Feature|Nudging Stacked Wild Totems|Nudging Wilds Feature|Outgrow Feature|Player's Choice Free Games Bonus|Prize Accumulator|Prize Disk Feature|Pyramid Free Spins|Re-Spin Feature|Re-spin Feature|ReSpin Feature|Reel Boost Feature|Reel Locking|Reel Touch Bonus|Renewal Feature|Respin Feature|Retriggering Free Games|Roaming Wilds|Shadow Masquerade Free Games|Special Free Game Bonus|Split Symbol Feature|Stacked Symbols Feature|Stinkin' Rich Keys to Riches Bonus|Streaking Stacks Feature|Super Free Games|Super Free Spins|Super Wheel Mania Feature|Switch-A-Roo Bonus|Symbol Pays Feature|Symbol Replacement|Symbols Removed Free Spins|Tiger Free Games|Timeless Romance Feature|Ultimate X|Ultimate X Free Games|Unlimited Free Games|Upgrade Feature|Variable Free Games|Vegas Free Games Feature|White Falls Free Spins|White Falls Super Free Spins|Wild Boost Feature|Wild Connections|Wild Multipliers|Wild Symbols|Wilds Meter|Win Multiplier|Winner's Choice 2 Mystery Bonus|Winner's Wave|Zodiac Free Games|bonus pick|bonus wilds|free games|free games bonus|free games feature|free spins|free spins bonus|free_games|free_games_bonus|free_spins|free_spins_bonus|increasing multiplier|locking wilds|move-expand stacks|multiplier feature|symbol replacement|wild boost feature|win multiplier""".split("|"),

    "Gamble / Double Up": """Double Up|Double Up/Gamble|Force Double Up|Gamble|Gamble Feature|Gamble/Double Up|Gamble/Double Up Feature|double_up|gamble/double up|gamble_feature""".split("|"),

    "Hold & Spin / Lock / Collect": """Blue Spark|Boingy Beans Feature|Bubble Burst Bonus|Character Upgrade|Collect Feature|Collector Feature|Connect & Collect Feature|Connect and Collect|Frame Locking|Gold Blast bonus|Golden Pepper Feature|High Stakes Boost Mode|Hold & Spin Bonus|Hold 'N Spin Feature|Hold and Spin|Hold and Spin Bonus|Instant Collect Feature|Locked Block Bonus|Money Comb Bonus|Morph Feature|Multi-Level Bonus|Mystery Credits Bonus|Power Quads Bonus|Prosperity Feature|Scatter Spins Feature|Silver Skies Bonus|Super Blocks Symbol Feature|dragon_cave_bonus|Hold & Respin Feature|Hold & Spin|Hold & Spin Feature|Hold and Respin Bonus|Hold-n-Spin Feature|Lock & Respin|Lock & Respin Bonus|Lock & Respin feature|Lock and Respin|Treasure Hold & Respin""".split("|"),

    "Interactive Bonus Games": """1st and 10 Bonus|Are you psychic? Bonus|Bad Play Feature|Big Bang Bonus|Big Bang and Spin Boost Bonus|Big City Bonus|Bobber Bonus Classic|Bonus Bet|Bonus Cards|Bonus Feature Trigger|Bonus Game|Bonus Games|Bonus Hands|Bonus Nudges Until It Pays Feature|Bonus Round|Bonus Trigger|Bubble Bonus|Buoy Bonus|Buoy Bonus Deluxe|Buy A Bonus|Buy a Pay Bonus|Cash Reels Bonus|Combo Reels|Credit Playoff|Diamond Bonus|Dice Roll Mechanic|Drum Boost Feature|Dynamic Player Panels (DPP)|Fire Opals Bonus|Flying Cow Bonus|Glorious Ingot Bonus|Golden Rooster Bonus|Hardway Pays|Hit Picks Feature|Hold and Split Bonus|Hot Poker Dice Bonus|Hot Roll Bonus|Hot Roulette Bonus|Hyper Bonus Feature|Ignite Feature|In-A-Row Bonus|Interactive Bonus|Lobster Buoy Bonus|Lucky Sky Bonus|Max Bet Eligibility|Maximum Bet Bonus|Money Ball Bonus|Money Balls Bonus|Money Mover Bonus|Multiway Bonus Bet|Pick an Urn Bonus|Pick and Slide Bonus|Pick-a-Pair|Pig a Prize|Pinball Bonus|Plus 2 Bonus|Power Up Feature|Selectable Bonus Games|Skill-Based Bonus|Skill-based Bonus|Spin Boost Bonus|Stellar Spins Bonus|Sticky Feature|Treasure Chest Bonus|Trigger Tile Functionality|Trigger-based Bonus|Weather Beacon Bonus|bonus bet|bonus game|bonus games|bonus_game|bonus_games|bonus_reel|bonus_symbol|hot bingo feature|Double Reels|Bartender Selection|Bonus Selection|Carnival Games Bonus|Crystal Fortunes Feature|DragonSphere Pick Bonus|Draw Poker|Fortune Pick Bonus|Gold Lantern Bonus|Hot Hit Pick Bonus|Joy & Luck Bonus Select|Multidenomination|Peach and Rich Pick Bonus|Pick Feature|Player Choice Bonus Feature|Risk/Reward|Second Chance Pick Bonus|Trash for Cash Pick Bonus|Zodiac Bonus Select|multi_denomination""".split("|"),

    "Multipliers": """Alien Wheel Bonus|Back-to-Back Multiplier|Bet Multiplier|Bonus Multiplier|Boutique Bonus|Buy-a-Pay bonus feature|Center Position Multiplier|Configurable Max Bet|Dice Roll Multiplier|Discovery Feature|Double Big Wheel Feature|Enchanted Multipliers Wheel Feature|Enchanted Window Bonus|FeverMode|Fortune X Multiplier Feature|Gem Multiplier Feature|Glorious Bonus|Haywire Bonus Feature|Haywire Feature|Haywire Multipliers Feature|I Dream of... Bonus|Increasing Multipliers|King Catch Bonus|Line Purchase Mechanic|Lucky Lights Feature|Lucky Spin|Lucky Suit Feature|Lucky Win Multipliers|Magic Diamond Feature|Major Nelson in Orbit! Bonus|Major Nelson's Bachelor Pad Bonus|Masquerade Multiplier Bonus|Multiplier|Multiplier Accumulation|Multiplier Bubble Feature|Multiplier Distribution|Multiplier Symbol|Multiplier Symbols|Multiplier Wheel|Multiplier Wheel Feature|Multiplier grid feature|Multipliers|Mystery Multipliers|Overlapping Multipliers|Prism Feature|Red Lantern Bonus|Repeating Win Multiplier|Repeating Win Multipliers|Rift Credits Feature|Row Multiplier|Sub-Symbol Multiplier|Super Times Pay Wheel Bonus Feature|Symbol Multiplier|Tap House Bonus|Terror Dog Multiplier|The Player's Choice Bonus|Trash for Cash Feature|Treasure Track|Ultimate X Gold Feature|Volatile Bonus|Xtra Line feature|configurable max bet|dice roll multiplier|double big wheel|lucky suit|multiplier|multiplier adjustment|multipliers|pick_bonus""".split("|"),

    "Mystery / Random Awards": """Coyote Moon Bonus|Eternal Pot Feature|Mystery Bill Feature|Pick A Reel Feature|Slimer Ballroom Busters Bonus|Reel Nudge""".split("|"),

    "Nudges / Reel Stops": """Reel Nudge""".split("|"),

    "Other Mechanic": """8-Game Counter|Any Adjacent Pays|Auto Hold|Auto Play|Auto-Play|AutoPlay|Autoplay|Autoplay Feature|Autplay|Base Game Selective Replacement Feature|Bet Levels|Big 5 Tracker Feature|Big Bigger Biggest Feature|Blackout Pays|Boom Pay|Bubble Boost|Bubble Boost Feature|Bubble Feature|Capture Feature|Cheerleader Feature|Classic Spinning-Reel Action|Configurability|Configurable Bet|Cost to cover|Custom Bet|Custom Bet Configuration|Diamond Tide Feature|Dice Interaction|Door Feature|Double Feature|Double Symbols|Dragon Feature|Dynamic Reel Height|Ecto-1 Symbol Swap|Elephants vs Tigers Feature|Extra Life|Extreme Stacks Feature|Feature|Feature Upgrade|Feature Upgrades|Fever Mode|Forced Bet Game|Forced Multiple Denomination|Fortune Rocket Feature|Fuzzy Factory|Graduating Payback|Graduating Payback Percentage|Growing WD Symbol|Halo Reels|Hyper Hits Feature|Independent Reels|Key Feature|King Reels Feature|King Stacks Feature|Lightning Trigger|Magnetic Reels|Magnetic Reels Feature|Matching Colored Bubbles|Megabucks Eligibility|Mirror Feature|Moneyball Wins|Moneyballs Feature|Multi-Denom|Multi-Denomination|Multi-Denomination Support|Multi-Game Feature|Multi-Play|Multi-Round Gameplay|Multi-denomination|Multi-play|MultiWay|MultiWay Wins|MultiWay game|MultiWayXtra|Multiple Denomination|Multiway|Multiway Wins|Nudging Feature|Picture Pay|Piñata Feature|Player Selectable Cashout|Prize Track|Quick Cash|REELdepth|RTP|RTP Configuration|Reel Multiplay|ReelStripSelection|Repeat Pays Feature|Rock Star Spin|Selective Stacking|Shopping Spree Feature|Split Symbols|Stacked Symbols|Stacked symbols|Super Feature|Super Features and Mega Feature|Super Stacks Feature|Symbol Replacement Feature|Symbol Upgrade|Three Kings Deluxe Feature|Tiger Color Change Feature|Tournament|Treasure Zone|Triple Reels|Triple Reels Feature|Tumble Mechanism|Tumbling Reels Feature|Watch and Win Feature|Win What You See|Win Zone Activator|Wu Da Jiang Tracker Feature|Yin-Yang Symbol|autoplay|autplay|big bigger biggest feature|bubble_boost|configurable bet|configurable_max_bet|feature|fixed max bet|king stacks feature|multi-denomination|multi-game|multi-play|multiple denomination|multiway|paytable_driven|paytables|stacked symbols""".split("|"),

    "Pick / Choice Bonus": """Bartender Selection|Bonus Selection|Carnival Games Bonus|Crystal Fortunes Feature|DragonSphere Pick Bonus|Draw Poker|Fortune Pick Bonus|Gold Lantern Bonus|Hot Hit Pick Bonus|Joy & Luck Bonus Select|Multidenomination|Peach and Rich Pick Bonus|Pick Feature|Player Choice Bonus Feature|Risk/Reward|Second Chance Pick Bonus|Trash for Cash Pick Bonus|Zodiac Bonus Select|multi_denomination|pick feature""".split("|"),

    "Progressive Jackpot": """2-Level Forced Progressive|4-Level Progressive|5-Level Forced Progressive|6-Level Game Controlled Progressive|Adjacency Pays + Progressives Bet|Blackout Pay/Progressive Pay|Boom Wheel|Boost Feature|Cherished Bonus Select|Coin Boost and Jackpot Super Feature|Coin Boost and Jackpot and Reel Boost Mega Feature|Collection Bonus Feature|Countdown Jackpots|Crystal Choice|Double Shot Bonus|EGM Bank Progressive Feature|Empress Treasure Tracker|Eternal Link Lock & Respin Gold|Eternal Link Lock & Respin Red|Expanding Progressives Feature|Extra Chance Respin Feature|Extra Jackpot Chance|Forced Game Controlled Progressive|Forced Progressive|Fortune Coin Feature|Fortune Pick|Galactic Jackpot Bonus|Game Controlled Forced Progressive|Game Controlled Progressive|Gold Bet|Gold Party Bonus|Golden Dice Progressives|Golden Dice Progressives Feature|Happy Lucky Bonus|Hot Roll Golden Dice Bonus|JACKPOT Feature|Jackpot|Jackpot Bonus|Jackpot Bonus Trigger|Jackpot Boost|Jackpot Chance Bonus|Jackpot Collection|Jackpot Feature|Jackpot Increase|Jackpot Pick Bonus|Jackpot Playoff|Jackpot Playoffs|Jackpot Wheel|Jackpot Wheel Bonus|Jackpot Wilds Feature|Jackpots|Keys to Riches Bonus|Link Progressive|Linked Progressive|Linked Progressive Jackpot|Lucky Pick Jackpots|Magic Wheel Bonus|Match 3 Bonus|Mega Feature|Multi-Level Progressives|Multiple Jackpots|Multiple Progressives|Multiply Feature|Mystery Jackpot|Mystery Panther Feature|Mystery Progressive|Mystery Progressives Feature|Mystery-driven Progressives|Nudging Wilds / Jackpot Wilds / Expanded Reels Mega Feature|Pegglicious Progressive|Pick Bonus|Pick and Match|Prize Boost Bonus|Progressive|Progressive Award|Progressive Bonus|Progressive Compatibility|Progressive Feature|Progressive Free Games|Progressive Free Games Feature|Progressive Game|Progressive Increment|Progressive Jackpot|Progressive Jackpots|Progressive Levels|Progressive Link|Progressive Mechanic|Progressive Meter|Progressive Pick Bonus|Progressive Pick Feature|Progressive Re-Spin|Progressive Reconciliation|Progressive Support|Progressive Type|Progressive and Jackpot|Progressives|Pyramid Progressives Tracker Feature|Random Multiplier|Re-Spins|Reel Boost and Jackpot Super Feature|Reel Lock Progressives|Stack Attack Progressives|Standalone Progressive|Standalone or Linked Progressive|Super Features|Symbol Driven Progressives|Top Award Winning Scenario|Treasure Hold & Respin|Wheel Bonus|Wild Symbol Movement|Wild Symbol Payout|coin_bonus|jackpot bonus|jackpot_bonus|jackpot_feature|jackpot_wheel|linked_progressive_setup|magic wheel bonus|progressive|progressive feature|progressive jackpot|progressive levels|progressive mechanic|progressive reset|progressive_feature|progressive_jackpot|random multiplier|reel lock progressives""".split("|"),

    "Respins / Retriggers": """+1 Spin Symbol|Acorn Feature|Babylon Re-Spin Bonus|Big Money Bonus|Cash Eruption|Chameleon Transformations with Re-spin|Cherished Respin Bonus|Gold Respins Feature|Golden Respins Feature|Hold & Respin Feature|Hold & Spin|Hold & Spin Feature|Hold and Respin Bonus|Hold-n-Spin Feature|Hot Hit Respin Bonus|Lock & Respin|Lock & Respin Bonus|Lock & Respin feature|Lock and Respin|Mystery Feature|Player's Choice ReSpin|Rapid Reload Bonus|Re-Spin & Re-Pay Feature|Re-spinning Bonus|ReSpin|ReSpin N'Win Feature|Red Hot Respin Feature|Respin|Respin Bonus|Respins|Rising Respins|Second Chance Feature|Second Chance Stacks|Splendid Ingot Bonus|Super Rich Re-Spins Bonus|Zodiac Respin Bonus|respin""".split("|"),

    "Scatter / Symbol Triggers": """Bar-B-Que Blitz Bonus|Disk Trigger Mechanism|Free Storm Scatter Bonus|Golden Empress Scatter Feature|Scatter Boost Feature|Scatter Bucks Bonus|Scatter Feature|Scatter Initiated Bonus|Scatter Pay|Scatter Pays|Scatter Symbol|Scatter Symbols|Scatter Trigger|Scatter Wins|Smash Bonus|scatter bucks feature|scatter prize|scatter win condition|scatter_pay|scatter_symbol""".split("|"),

    "Side Bet / Wagering": """Bet Mechanism|Bonus Feature|Dream Card|Extra Hands Feature|House Advantage|Maxbet Bonus|Multiway Wagering|Tie Bet Payout|bonus feature|side_bet_feature""".split("|"),

    "Ways / Paylines / Win Patterns": """100x Bonus|2-for-1 wagering|243 Ways to Win|Bamboo Feature|Bettor Chance Way|Buy-A-Pay|Buy-a-Pay|Buy-a-pay|Easy Bet|Feature Trigger|Hot Roll Golden Dice|Line Boost Feature|Line Pays|Luck Zone Prizes|Multi-WayXtra|MultiWay Mechanic|MultiWay Mechanics|Multiplay Feature|Multiway Bet|Multiway Bonus Bet Feature|Payline|Paylines|Side Bets|Triple Top Dollar Bonus|Tumbling Reels|Ways to Win|buy-a-pay|line win condition|pattern_pays|payline|side bets|ways_to_win""".split("|"),

    "Wheel Bonus": """5 Wheel Bonus|Big Wheel Bonus|Bonus|Bonus Wheel|Bonus Wheel Feature|Credits Wheel Feature|Gateway Bonus|Gold Wheel Bonus|Jungle Wheel Bonus|La Partage Rule|Reveal-A-Wheel Feature|Roulette|Roulette Bonus|Triple Big Wheel Bonus|Wheel Feature|Wheel Nudge|bonus_wheel|wheel_bonus""".split("|"),

    "Wild Symbols and Wild Reels": """Added Wilds|Base Game Bonuses|Bettor Chance|Bonus Trigger Wild Collection Feature|Buddha Collection|Buffalo Stamping|COIN Feature|Cannon Animation|Cash Wilds Feature|Coin Feature|Collect and Upgrade|Crashing Wilds|Crashing Wilds Feature|Delayed Expanding Wild Feature|Delayed Expanding Wilds|Delayed Wilds|Detonating Wilds|Double Symbols and 5x Wilds|Double Tiger Feature|Dragon Wild Stacks|Dragons vs Pandas Feature|Dragonsphere Feature|Empresses vs Emperors Feature|Enduring Wilds|Expanding Wild Feature|Expanding Wild feature|Expanding Wilds|Extra Wild Feature|Extra Wilds Feature|Fortune Rooster Feature|Fuzzy Wilds Feature|Giant Wilds|Gift Box Feature|Gold Dragon Feature|Gold in Motion Feature|Guaranteed Cash Wilds Feature|Haywire!|Hinge Wilds Feature|Jumping Wilds|King Stacks|Lucky Spot Feature|Metered Wild Stacks|Multiple Play Options|Multiplier Effect|Multiplier Wild|Multiplier Wilds|Mystery Triggered Super Dragon Expanding Wild|Nudging Wilds|Phoenix Wild|Poker Boost Feature|Portal Lock|Quick Collect Feature|Random Wild|Random Wild Base Game Feature|Random Wild Feature|Random Wilds Collection Feature|Random Wilds Feature|Reel Wild Mechanic|Rift Zone Feature|Roaming Multiplier Wilds|Roaming Wild|Sally's Bonus Wilds|Silver Skies Feature|Slimer Wild|Special Wild Symbol|Special Wilds|Stack Attack Feature|Stacked Wilds|Stacked Wilds Feature|Stars of Wealth Feature|Sticky Wilds|Sub-Symbol Activation|Supercharged Haywire!|Surrounding Wild|Symbol Transformation|Upgrading Symbols|Wealth Feature|Wild Bubble Feature|Wild Card|Wild Card Feature|Wild Card Rule|Wild Cards|Wild Drop Feature|Wild Expansion Feature|Wild Feature|Wild Generation|Wild Multiplier|Wild Multiplier Feature|Wild Paths Feature|Wild Population Feature|Wild Reel Feature|Wild Reels|Wild Reels Feature|Wild Shapes|Wild Spins Feature|Wild Stacks|Wild Stays Charges Then Pays|Wild Stays Feature|Wild Stays Until it Pays|Wild Stays, Nudges and Pays|Wild Switch Feature|Wild Symbol|Wild Symbol Activation|Wild Symbol Multiplier|Wild Symbol Substitute|Wild Symbol Substitution|Wild Symbols Nudge|Wild Transformations|Wild and Scatter Pays|Wrecking Ball Feature|bonus_bet|expanding_wild|gigantic wilds|multiplier wilds feature|persistent_wilds|random wilds feature|random_wilds_feature|stacked wilds|sticky_wilds|surrounding wild|wild|wild reels feature|wild symbol|wild symbol multiplier|wild_boost|wild_bubble_feature|wild_reel_feature|wild_symbol|wild_symbols""".split("|"),
}

MECHANIC_DROP_KEYS = []

In [0]:
def normalize_label_key(value):
    return " ".join(str(value).strip().lower().split())


def build_mapping_df(
    raw_definition_table,
    raw_name_column,
    raw_key_column,
    overrides,
    drop_keys,
    mapping_type,
):
    raw_rows = (
        spark.table(raw_definition_table)
        .select(raw_name_column, raw_key_column)
        .where(F.col(raw_key_column).isNotNull())
        .groupBy(raw_key_column)
        .agg(F.min(raw_name_column).alias("raw_label"))
        .collect()
    )

    # Use normalized keys so the override matching follows the same
    # case and whitespace rules as normalize_label_key().
    labels_by_key = {
        normalize_label_key(row[raw_key_column]): row["raw_label"]
        for row in raw_rows
    }
    known_keys = set(labels_by_key)

    # overrides format:
    # {
    #     "Canonical label": ["raw label 1", "raw label 2"],
    #     "Another canonical label": ["raw label 2"],
    # }
    # A raw label listed under multiple canonical labels produces
    # multiple mapping rows.
    targets_by_raw_key = {}

    for canonical_label, raw_labels in overrides.items():
        if canonical_label is None or not str(canonical_label).strip():
            raise ValueError(
                f"{mapping_type}: canonical labels cannot be blank."
            )

        clean_canonical_label = str(canonical_label).strip()

        if isinstance(raw_labels, str):
            raw_labels = [raw_labels]

        for raw_label in raw_labels:
            raw_key = normalize_label_key(raw_label)

            if not raw_key:
                raise ValueError(
                    f"{mapping_type}: raw labels in merge lists "
                    "cannot be blank."
                )

            targets = targets_by_raw_key.setdefault(raw_key, [])
            target_key = normalize_label_key(clean_canonical_label)

            if all(
                normalize_label_key(existing) != target_key
                for existing in targets
            ):
                targets.append(clean_canonical_label)

    normalized_drop_keys = {
        normalize_label_key(raw_key)
        for raw_key in drop_keys
    }

    mapped_keys = set(targets_by_raw_key)

    unknown_keys = (mapped_keys | normalized_drop_keys) - known_keys
    if unknown_keys:
        raise ValueError(
            f"{mapping_type}: mapping keys not found in raw definitions: "
            f"{sorted(unknown_keys)}"
        )

    conflicting_keys = mapped_keys & normalized_drop_keys
    if conflicting_keys:
        raise ValueError(
            f"{mapping_type}: labels cannot be both mapped and dropped: "
            f"{sorted(conflicting_keys)}"
        )

    mapping_rows = []

    for raw_key, raw_label in sorted(labels_by_key.items()):
        if raw_key in normalized_drop_keys:
            mapping_rows.append((
                raw_label,
                raw_key,
                None,
                "drop",
                mapping_type,
                RUN_ID,
                TAXONOMY_VERSION,
            ))
            continue

        # Unmapped labels keep their original label.
        canonical_labels = targets_by_raw_key.get(raw_key, [raw_label])

        for canonical_label in canonical_labels:
            action = (
                "keep"
                if normalize_label_key(canonical_label) == raw_key
                else "merge"
            )

            mapping_rows.append((
                raw_label,
                raw_key,
                canonical_label,
                action,
                mapping_type,
                RUN_ID,
                TAXONOMY_VERSION,
            ))

    mapping_schema = T.StructType([
        T.StructField("raw_label", T.StringType(), False),
        T.StructField("raw_label_key", T.StringType(), False),
        T.StructField("canonical_label", T.StringType(), True),
        T.StructField("action", T.StringType(), False),
        T.StructField("mapping_type", T.StringType(), False),
        T.StructField("run_id", T.StringType(), False),
        T.StructField("taxonomy_version", T.StringType(), False),
    ])

    return spark.createDataFrame(mapping_rows, schema=mapping_schema)

theme_mapping_df = build_mapping_df(
    raw_definition_table=RAW_THEME_DEF_TABLE,
    raw_name_column="raw_theme_name",
    raw_key_column="raw_theme_name_key",
    overrides=THEME_CANONICAL_OVERRIDES,
    drop_keys=THEME_DROP_KEYS,
    mapping_type="theme",
)

mechanic_mapping_df = build_mapping_df(
    raw_definition_table=RAW_MECHANIC_DEF_TABLE,
    raw_name_column="raw_mechanic_name",
    raw_key_column="raw_mechanic_name_key",
    overrides=MECHANIC_CANONICAL_OVERRIDES,
    drop_keys=MECHANIC_DROP_KEYS,
    mapping_type="mechanic",
)

(
    theme_mapping_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(THEME_MAPPING_TABLE)
)

(
    mechanic_mapping_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(MECHANIC_MAPPING_TABLE)
)

print(f"Theme mapping table: {THEME_MAPPING_TABLE}")
display(theme_mapping_df.orderBy("raw_label", "canonical_label"))

print(f"Mechanic mapping table: {MECHANIC_MAPPING_TABLE}")
display(mechanic_mapping_df.orderBy("raw_label", "canonical_label"))

In [0]:
# Optional approved definitions for merged canonical labels.
# Keys must exactly match the canonical labels in the mapping tables.

THEME_DEFINITION_OVERRIDES = {
    # "Canonical Theme": "Approved combined definition",
}

MECHANIC_DEFINITION_OVERRIDES = {
    # "Canonical Mechanic": "Approved combined definition",
}


def string_map_expression(overrides):
    if not overrides:
        return F.lit(None).cast(
            T.MapType(T.StringType(), T.StringType())
        )

    entries = []
    for key, value in overrides.items():
        entries.extend([F.lit(key), F.lit(value)])

    return F.create_map(*entries)


def add_canonical_definition(dictionary_df, label_column, overrides):
    override_map = string_map_expression(overrides)

    return (
        dictionary_df
        .withColumn(
            "_definition_override",
            F.element_at(override_map, F.col(label_column))
        )
        .withColumn(
            "canonical_definition",
            F.when(
                F.col("_definition_override").isNotNull(),
                F.col("_definition_override")
            )
            .when(
                F.size(F.col("source_definitions")) == 1,
                F.element_at(F.col("source_definitions"), 1)
            )
            .otherwise(F.lit(None).cast("string"))
        )
        .drop("_definition_override")
    )


# Read the saved raw assignments and approved mappings.
raw_theme_df = spark.table(RAW_THEME_ASSIGNMENTS_TABLE)
raw_mechanic_df = spark.table(RAW_MECHANIC_ASSIGNMENTS_TABLE)

theme_mapping_for_join_df = (
    spark.table(THEME_MAPPING_TABLE)
    .select(
        F.col("raw_label_key").alias("raw_theme_name_key"),
        F.col("canonical_label").alias("canonical_theme"),
        F.col("action").alias("theme_mapping_action"),
    )
)

mechanic_mapping_for_join_df = (
    spark.table(MECHANIC_MAPPING_TABLE)
    .select(
        F.col("raw_label_key").alias("raw_mechanic_name_key"),
        F.col("canonical_label").alias("canonical_mechanic"),
        F.col("action").alias("mechanic_mapping_action"),
    )
)

# Apply theme mappings. Dropped labels remain in the raw table but do not
# appear in canonical assignments.
theme_joined_df = raw_theme_df.join(
    theme_mapping_for_join_df,
    on="raw_theme_name_key",
    how="left",
)

unmapped_themes = theme_joined_df.where(
    F.col("theme_mapping_action").isNull()
).count()

if unmapped_themes:
    raise ValueError(
        f"{unmapped_themes} raw theme assignment rows have no mapping."
    )

canonical_theme_assignments_df = theme_joined_df.where(
    (F.col("theme_mapping_action") != "drop")
    & F.col("canonical_theme").isNotNull()
)

# Apply mechanic mappings.
mechanic_joined_df = raw_mechanic_df.join(
    mechanic_mapping_for_join_df,
    on="raw_mechanic_name_key",
    how="left",
)

unmapped_mechanics = mechanic_joined_df.where(
    F.col("mechanic_mapping_action").isNull()
).count()

if unmapped_mechanics:
    raise ValueError(
        f"{unmapped_mechanics} raw mechanic assignment rows have no mapping."
    )

canonical_mechanic_assignments_df = mechanic_joined_df.where(
    (F.col("mechanic_mapping_action") != "drop")
    & F.col("canonical_mechanic").isNotNull()
)

# Build theme dictionary summaries from the assigned game records.
theme_dictionary_df = (
    canonical_theme_assignments_df
    .groupBy("canonical_theme")
    .agg(
        F.sort_array(F.collect_set("raw_theme_name")).alias("raw_labels"),
        F.sort_array(
            F.collect_set("raw_theme_definition")
        ).alias("source_definitions"),
        F.countDistinct("raw_theme_name_key").alias("raw_label_count"),
        F.countDistinct("game_name_key").alias("game_count"),
    )
)

theme_dictionary_df = (
    add_canonical_definition(
        theme_dictionary_df,
        "canonical_theme",
        THEME_DEFINITION_OVERRIDES,
    )
    .withColumnRenamed("canonical_theme", "canonical_label")
    .withColumn("taxonomy_type", F.lit("theme"))
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("taxonomy_version", F.lit(TAXONOMY_VERSION))
)

# Build mechanic dictionary summaries.
mechanic_dictionary_df = (
    canonical_mechanic_assignments_df
    .groupBy("canonical_mechanic")
    .agg(
        F.sort_array(F.collect_set("raw_mechanic_name")).alias("raw_labels"),
        F.sort_array(
            F.collect_set("raw_mechanic_definition")
        ).alias("source_definitions"),
        F.countDistinct("raw_mechanic_name_key").alias("raw_label_count"),
        F.countDistinct("game_name_key").alias("game_count"),
    )
)

mechanic_dictionary_df = (
    add_canonical_definition(
        mechanic_dictionary_df,
        "canonical_mechanic",
        MECHANIC_DEFINITION_OVERRIDES,
    )
    .withColumnRenamed("canonical_mechanic", "canonical_label")
    .withColumn("taxonomy_type", F.lit("mechanic"))
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("taxonomy_version", F.lit(TAXONOMY_VERSION))
)

# Attach canonical definitions to the game-level assignments.
canonical_theme_assignments_df = (
    canonical_theme_assignments_df
    .join(
        theme_dictionary_df.select(
            F.col("canonical_label").alias("canonical_theme"),
            F.col("canonical_definition").alias(
                "canonical_theme_definition"
            ),
        ),
        on="canonical_theme",
        how="left",
    )
)

canonical_mechanic_assignments_df = (
    canonical_mechanic_assignments_df
    .join(
        mechanic_dictionary_df.select(
            F.col("canonical_label").alias("canonical_mechanic"),
            F.col("canonical_definition").alias(
                "canonical_mechanic_definition"
            ),
        ),
        on="canonical_mechanic",
        how="left",
    )
)

# Save canonical dictionaries and per-game assignments.
(
    theme_dictionary_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(THEME_DICTIONARY_TABLE)
)

(
    mechanic_dictionary_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(MECHANIC_DICTIONARY_TABLE)
)

(
    canonical_theme_assignments_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(CANONICAL_THEME_ASSIGNMENTS_TABLE)
)

(
    canonical_mechanic_assignments_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(CANONICAL_MECHANIC_ASSIGNMENTS_TABLE)
)

print(f"Theme dictionary: {THEME_DICTIONARY_TABLE}")
print(f"Mechanic dictionary: {MECHANIC_DICTIONARY_TABLE}")
print(f"Canonical theme assignments: {CANONICAL_THEME_ASSIGNMENTS_TABLE}")
print(f"Canonical mechanic assignments: {CANONICAL_MECHANIC_ASSIGNMENTS_TABLE}")

display(theme_dictionary_df.orderBy("canonical_label"))
display(mechanic_dictionary_df.orderBy("canonical_label"))

In [0]:
THEME_GAME_SCORES_TABLE = (
    f"{RES_PATH}.overview_ctc_line_reel_theme_game_scores_{TAXONOMY_VERSION}"
)
MECHANIC_GAME_SCORES_TABLE = (
    f"{RES_PATH}.overview_ctc_line_reel_mechanic_game_scores_{TAXONOMY_VERSION}"
)

canonical_themes_for_scoring_df = spark.table(
    CANONICAL_THEME_ASSIGNMENTS_TABLE
)

canonical_mechanics_for_scoring_df = spark.table(
    CANONICAL_MECHANIC_ASSIGNMENTS_TABLE
)

# Check that theme scores follow the requested 0–5 scale.
invalid_theme_scores = canonical_themes_for_scoring_df.where(
    F.col("raw_theme_score").isNotNull()
    & (
        (F.col("raw_theme_score") < 0)
        | (F.col("raw_theme_score") > 5)
    )
).count()

if invalid_theme_scores:
    raise ValueError(
        f"Found {invalid_theme_scores} theme scores outside the 0–5 range."
    )

# Mechanics are binary in this extraction: 0 = explicitly absent,
# 1 = present, null = unknown.
invalid_mechanic_scores = canonical_mechanics_for_scoring_df.where(
    F.col("raw_mechanic_score").isNotNull()
    & (~F.col("raw_mechanic_score").isin(0.0, 1.0))
).count()

if invalid_mechanic_scores:
    raise ValueError(
        f"Found {invalid_mechanic_scores} mechanic scores outside 0/1. "
        "Review the extracted mechanic score scale before aggregating."
    )

theme_game_scores_df = (
    canonical_themes_for_scoring_df
    .groupBy("game_name", "game_name_key", "canonical_theme")
    .agg(
        F.max("raw_theme_score").alias("theme_score"),
        F.sort_array(
            F.collect_set("raw_theme_name")
        ).alias("raw_theme_labels"),
        F.countDistinct("raw_theme_name_key").alias("raw_theme_label_count"),
        F.first(
            "canonical_theme_definition",
            ignorenulls=True
        ).alias("canonical_theme_definition"),
    )
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("taxonomy_version", F.lit(TAXONOMY_VERSION))
)

mechanic_game_scores_df = (
    canonical_mechanics_for_scoring_df
    .groupBy("game_name", "game_name_key", "canonical_mechanic")
    .agg(
        F.max("raw_mechanic_score").alias("mechanic_score"),
        F.sort_array(
            F.collect_set("raw_mechanic_name")
        ).alias("raw_mechanic_labels"),
        F.countDistinct(
            "raw_mechanic_name_key"
        ).alias("raw_mechanic_label_count"),
        F.first(
            "canonical_mechanic_definition",
            ignorenulls=True
        ).alias("canonical_mechanic_definition"),
    )
    .withColumn(
        "mechanic_presence",
        F.when(F.col("mechanic_score") > 0, F.lit(1))
        .when(F.col("mechanic_score") == 0, F.lit(0))
        .otherwise(F.lit(None).cast("int"))
    )
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("taxonomy_version", F.lit(TAXONOMY_VERSION))
)

(
    theme_game_scores_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(THEME_GAME_SCORES_TABLE)
)

(
    mechanic_game_scores_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(MECHANIC_GAME_SCORES_TABLE)
)

print(f"Theme scores: {THEME_GAME_SCORES_TABLE}")
print(f"Mechanic scores: {MECHANIC_GAME_SCORES_TABLE}")

display(theme_game_scores_df.orderBy("game_name", "canonical_theme"))
display(mechanic_game_scores_df.orderBy("game_name", "canonical_mechanic"))

In [0]:
import re

# Reload the source so this cell can run even if earlier DataFrames
# are no longer in the notebook session.
candidate_source_df = (
    spark.table(SOURCE_TABLE)
    .withColumnRenamed("themes", "raw_themes")
)
candidate_source_df = (
    candidate_source_df
    .withColumn(
        "game_name_key",
        F.regexp_replace(
            F.lower(F.trim(F.col("game_name"))),
            r"\s+",
            " "
        )
    )
)

# The join key must identify exactly one source row per game.
duplicate_source_keys_df = (
    candidate_source_df
    .groupBy("game_name_key")
    .count()
    .where(F.col("count") > 1)
)

duplicate_source_key_count = duplicate_source_keys_df.count()

if duplicate_source_key_count:
    print(
        f"Warning: {duplicate_source_key_count} game_name_key values "
        "have multiple source rows. Keeping all rows for now."
    )


def make_feature_name(prefix, label, suffix):
    """Create a readable, stable column name from a canonical label."""
    slug = re.sub(r"[^a-z0-9]+", "_", label.lower()).strip("_")
    return f"{prefix}_{slug}_{suffix}"


def check_feature_name_collisions(labels, prefix, suffix):
    """Stop if different labels would produce the same output column."""
    names_to_labels = {}

    for label in labels:
        feature_name = make_feature_name(prefix, label, suffix)
        names_to_labels.setdefault(feature_name, []).append(label)

    collisions = {
        feature_name: source_labels
        for feature_name, source_labels in names_to_labels.items()
        if len(source_labels) > 1
    }

    if collisions:
        raise ValueError(
            f"Canonical labels produce duplicate {prefix} column names: "
            f"{collisions}"
        )


# Read canonical score tables created in Cell 8.
theme_scores_source_df = spark.table(THEME_GAME_SCORES_TABLE)
mechanic_scores_source_df = spark.table(MECHANIC_GAME_SCORES_TABLE)
canonical_theme_names_df = (
    theme_scores_source_df
    .groupBy("game_name_key")
    .agg(
        F.sort_array(F.collect_set("canonical_theme")).alias("themes")
    )
)
theme_labels = sorted(
    row["canonical_theme"]
    for row in (
        theme_scores_source_df
        .select("canonical_theme")
        .where(F.col("canonical_theme").isNotNull())
        .distinct()
        .collect()
    )
)

mechanic_labels = sorted(
    row["canonical_mechanic"]
    for row in (
        mechanic_scores_source_df
        .select("canonical_mechanic")
        .where(F.col("canonical_mechanic").isNotNull())
        .distinct()
        .collect()
    )
)

# Check column names before pivoting.
check_feature_name_collisions(theme_labels, "theme", "score")
check_feature_name_collisions(mechanic_labels, "mechanic", "score")
check_feature_name_collisions(mechanic_labels, "mechanic", "present")

# Expand theme scores into one column per canonical theme.
if theme_labels:
    theme_wide_raw_df = (
        theme_scores_source_df
        .groupBy("game_name_key")
        .pivot("canonical_theme", theme_labels)
        .agg(F.max("theme_score"))
    )

    theme_wide_df = theme_wide_raw_df.select(
        "game_name_key",
        *[
            F.col(f"`{label.replace('`', '``')}`").alias(
                make_feature_name("theme", label, "score")
            )
            for label in theme_labels
        ],
    )
else:
    theme_wide_df = candidate_source_df.select("game_name_key").limit(0)

# Expand mechanic scores and presence into separate columns.
if mechanic_labels:
    mechanic_score_wide_raw_df = (
        mechanic_scores_source_df
        .groupBy("game_name_key")
        .pivot("canonical_mechanic", mechanic_labels)
        .agg(F.max("mechanic_score"))
    )

    mechanic_presence_wide_raw_df = (
        mechanic_scores_source_df
        .groupBy("game_name_key")
        .pivot("canonical_mechanic", mechanic_labels)
        .agg(F.max("mechanic_presence"))
    )

    mechanic_wide_df = mechanic_score_wide_raw_df.select(
        "game_name_key",
        *[
            F.col(f"`{label.replace('`', '``')}`").alias(
                make_feature_name("mechanic", label, "score")
            )
            for label in mechanic_labels
        ],
    )

    mechanic_presence_wide_df = mechanic_presence_wide_raw_df.select(
        "game_name_key",
        *[
            F.col(f"`{label.replace('`', '``')}`").alias(
                make_feature_name("mechanic", label, "present")
            )
            for label in mechanic_labels
        ],
    )

    mechanic_wide_df = mechanic_wide_df.join(
        mechanic_presence_wide_df,
        on="game_name_key",
        how="left",
    )
else:
    mechanic_wide_df = candidate_source_df.select("game_name_key").limit(0)

# Keep the extracted game facts and attach the expanded canonical features.
candidate_game_df = (
    candidate_source_df
    .join(theme_wide_df, on="game_name_key", how="left")
    .join(canonical_theme_names_df, on="game_name_key", how="left")
    .join(mechanic_wide_df, on="game_name_key", how="left")
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("taxonomy_version", F.lit(TAXONOMY_VERSION))
)
print(f"Candidate rows: {candidate_game_df.count()}")
print(f"Theme score columns added: {len(theme_labels)}")
print(f"Mechanic score/presence pairs added: {len(mechanic_labels)}")
print("Candidate dataset preview:")
display(candidate_game_df.limit(10))

In [0]:
from functools import reduce

# Check that the candidate has one row per game.
candidate_count = candidate_game_df.count()

duplicate_candidate_keys_df = (
    candidate_game_df
    .groupBy("game_name_key")
    .count()
    .where(F.col("count") > 1)
)

duplicate_candidate_count = duplicate_candidate_keys_df.count()

# Confirm every source game is present in the candidate.
source_keys_df = candidate_source_df.select("game_name_key").distinct()
candidate_keys_df = candidate_game_df.select("game_name_key").distinct()

missing_candidate_games_df = source_keys_df.join(
    candidate_keys_df,
    on="game_name_key",
    how="left_anti",
)

missing_candidate_count = missing_candidate_games_df.count()

# Check theme score columns are within the 0–5 range.
invalid_theme_score_conditions = [
    F.col(make_feature_name("theme", label, "score")).isNotNull()
    & (
        (F.col(make_feature_name("theme", label, "score")) < 0)
        | (F.col(make_feature_name("theme", label, "score")) > 5)
    )
    for label in theme_labels
]

invalid_theme_score_count = (
    candidate_game_df
    .where(reduce(lambda left, right: left | right, invalid_theme_score_conditions))
    .count()
    if invalid_theme_score_conditions
    else 0
)

# Check mechanic scores and presence values are binary or null.
invalid_mechanic_conditions = []

for label in mechanic_labels:
    score_col = make_feature_name("mechanic", label, "score")
    presence_col = make_feature_name("mechanic", label, "present")

    invalid_mechanic_conditions.extend([
        F.col(score_col).isNotNull()
        & (~F.col(score_col).isin(0.0, 1.0)),
        F.col(presence_col).isNotNull()
        & (~F.col(presence_col).isin(0, 1)),
    ])

invalid_mechanic_count = (
    candidate_game_df
    .where(reduce(lambda left, right: left | right, invalid_mechanic_conditions))
    .count()
    if invalid_mechanic_conditions
    else 0
)

print(f"Source rows: {candidate_source_df.count()}")
print(f"Candidate rows: {candidate_count}")
print(f"Duplicate candidate game keys: {duplicate_candidate_count}")
print(f"Source games missing from candidate: {missing_candidate_count}")
print(f"Rows with invalid theme scores: {invalid_theme_score_count}")
print(f"Rows with invalid mechanic scores/presence: {invalid_mechanic_count}")

# Show specific source games that did not make it into the candidate, if any.
if missing_candidate_count:
    display(missing_candidate_games_df)

if duplicate_candidate_count:
    print(
        f"Warning: keeping {duplicate_candidate_count} duplicated "
        "game_name_key values for now."
    )

if missing_candidate_count or invalid_theme_score_count or invalid_mechanic_count:
    raise ValueError(
        "Candidate QA failed. Review missing games or invalid scores before saving."
    )

# Save the candidate and publish the final game-level dataset.
(
    candidate_game_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(CANDIDATE_GAME_TABLE)
)

(
    candidate_game_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FINAL_GAME_TABLE)
)

print(f"Candidate dataset saved to: {CANDIDATE_GAME_TABLE}")
print(f"Final dataset published to: {FINAL_GAME_TABLE}")

display(
    spark.table(FINAL_GAME_TABLE)
    .orderBy("game_name")
    .limit(20)
)